In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

# 数据集

In [2]:
words = open("names.txt", 'r').read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [18]:
# 老师的实现，更像个滚动窗口
block_size = 3  # context length: 预测下一个字符时所使用的上文长度
X,Y = [], []
for w in words[:5]:
    print(w)
    context = [0]*block_size  # context 用来存放当前上下文的变量
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} ---> {ch}")
        context = context[1:] + [ix]  #上下文向右滑动一位 crop and append
X = torch.tensor(X)
Y= torch.tensor(Y)
print(X.shape, Y.shape, X.dtype, Y.dtype)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
torch.Size([32, 3]) torch.Size([32]) torch.int64 torch.int64


In [14]:
# 构建用于MLP的数据集 这里输入变成3个字符 输出下一个字符 即：用前三个字符预测下一个字符

# 我的实现，其实结果是一样的，但是给人的含义不同
block_size = 3  # context length: 预测下一个字符时所使用的上文长度
X,Y = [], []
for w in words[:5]:
    print(w)
    context = ['.', '.', '.'] + list(w) + ['.']  
    # 这里这个block_size=3 其实就是输入的窗口长度，加上一个输出，其实整体的窗口长度是4，为了防止出现长度小于4的样本，这里直接对所有样本全部填充了...（这里的 . 字符其实是空白字符，占位用的）
    for i in range(len(context)-block_size):
        in_chs = context[i:i+block_size]
        out_ch = context[i+block_size]
        in_idx = [stoi[ch] for ch in in_chs]
        out_idx = stoi[out_ch]
        X.append(in_idx)
        Y.append(out_idx)
        print(f"{''.join(in_chs)} ---> {out_ch}")
X = torch.tensor(X)
Y= torch.tensor(Y)
print(X.shape, Y.shape)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
torch.Size([32, 3]) torch.Size([32])


# 构建查找表

原始论文是 1.7w个单词，嵌入到30维的空间里，即： 每个单词是一个30维的向量表示

这里因为全部的词表只有27个字符，因此简单点，先使用2维表示

注意，在之前的Bigram/单个线性层中，可以理解为：一个字符是由一个27维的向量表示的 

In [19]:
C = torch.randn((27,2))
C

tensor([[-1.0995, -2.1420],
        [-1.1483, -0.3677],
        [-2.6622, -1.2882],
        [-1.7438, -1.8135],
        [ 0.6581, -0.6228],
        [-0.6300, -0.5383],
        [-0.2353,  0.6547],
        [ 0.7970, -1.0498],
        [ 0.7984, -0.3832],
        [-0.5210,  0.2150],
        [ 1.2209,  0.8330],
        [ 0.2303,  1.7591],
        [ 0.6781, -0.6105],
        [ 1.2900,  0.1468],
        [ 0.2859,  1.0927],
        [ 1.3411,  0.6629],
        [ 0.0248,  0.1640],
        [-0.8785, -1.0223],
        [-0.2280, -0.2069],
        [-0.7846, -0.0809],
        [ 1.8169,  0.2405],
        [-0.9264, -1.4094],
        [-0.2677,  0.9994],
        [-0.0547, -0.1928],
        [-1.1092,  2.9970],
        [-0.2838, -0.1218],
        [-0.0759,  0.0406]])